# Validation sentences and masked inside charts

Sample sentences from one fixed grammar, mask the final token while running the inside algorithm, and retain the two non-root latent levels for probing.

In [ ]:
import os
import random

import numpy as np
import torch
import time
from Inside_outside import (
    extract_binary_rules_as_tensors,
    extract_ternary_rules_as_tensors,
    generate_grammar,
    generate_grammar_2,
    generate_rules_double_new,
    parallel_inside_masked,
    sample_from_grammar,
)

In [ ]:
V = 8
F = 1 / 4
DEPTH = 3
RULE_SEED = 12345678
SAMPLE_SEED = 100
N_SAMPLES = int(os.environ.get("N_SAMPLES", 100))
GENERATION_ETA = 0.0
PARSING_ETA = V  # Always parse under the general binary/ternary grammar.
# Keeping every four-level chart as well as the probing representation roughly
# doubles memory use. Leave this False unless the original charts are needed too.
KEEP_FULL_CHARTS = False
STORAGE_DTYPE = torch.float64

## Construct the fixed grammar and its rule tensors

In [ ]:
n = V
m_2 = int(F * V)
m_3 = int(F * V**2)
def rule_probabilities(eta):
    p_3 = 1 / (m_2 * eta + m_3)
    p_2 = eta * p_3
    return p_2, p_3

generation_p_2, generation_p_3 = rule_probabilities(GENERATION_ETA)
parsing_p_2, parsing_p_3 = rule_probabilities(PARSING_ETA)

# Reusing RULE_SEED gives both grammars exactly the same sampled rule support;
# only their binary/ternary probabilities differ. Under this codebase's
# convention, eta=0 means p_2=0 (pure ternary generation).
generation_rules, _ = generate_rules_double_new(
    V, n, m_2, m_3, generation_p_2, generation_p_3, DEPTH, RULE_SEED
)
_, parsing_rules = generate_rules_double_new(
    V, n, m_2, m_3, parsing_p_2, parsing_p_3, DEPTH, RULE_SEED
)
sampling_grammar = generate_grammar(generation_rules)
tensor_grammar = generate_grammar_2(parsing_rules)

binary_by_layer = extract_binary_rules_as_tensors(tensor_grammar, V)
ternary_by_layer = extract_ternary_rules_as_tensors(tensor_grammar, V)
layer_names = sorted(
    binary_by_layer, key=lambda name: int(name.removeprefix("layer"))
)
tensor_bin = torch.stack([binary_by_layer[name] for name in layer_names])
tensor_ter = torch.stack([ternary_by_layer[name] for name in layer_names])

print(f"generation: eta={GENERATION_ETA:g}, p_2*m_2={generation_p_2*m_2:g}, p_3*m_3={generation_p_3*m_3:g}")
print(f"parsing:    eta={PARSING_ETA:g}, p_2*m_2={parsing_p_2*m_2:g}, p_3*m_3={parsing_p_3*m_3:g}")

## Boundary representation

For each latent level, sum the inside probabilities of all spans ending at the same location. This deliberately discards the span-start information and produces one length-8 vector per token boundary. The two returned tensors correspond to the two bottom-up non-root latent levels.

In [ ]:
def charts_to_boundaries(inside_tensor_list, sentence_length, dtype=STORAGE_DTYPE):
    """Return the two non-root levels as [token boundary, latent] tensors.

    inside_tensor_list[0] contains leaves, [1] and [2] contain the first
    and second bottom-up latent levels, and [DEPTH] contains the root.
    Trimming removes unused positions without losing sentence-chart entries.
    """
    boundary_levels = []
    for level_chart in inside_tensor_list[1:DEPTH]:
        trimmed = level_chart[:, : sentence_length + 1, : sentence_length + 1]
        # Sum over every possible start for each (end, latent) pair. End 0
        # cannot terminate a non-empty span, so omit it to align rows with tokens.
        collapsed = trimmed.sum(dim=1).transpose(0, 1)[1:]
        boundary_levels.append(
            collapsed.contiguous().to(device="cpu", dtype=dtype)
        )
    return tuple(boundary_levels)


def flatten_boundary_levels(boundary_levels):
    """Return the separate [token boundary, latent] features unchanged."""
    return tuple(boundary_levels)

## Generate the validation set

The call uses `mask_from=d`, so position `d - 1` (the final token) is replaced by a uniform terminal inside vector. Sentence sampling is reproducible and independent of the rule seed.

In [ ]:
time_1=time.time()
random.seed(SAMPLE_SEED)
np.random.seed(SAMPLE_SEED)
torch.manual_seed(SAMPLE_SEED)

sentences = []
inside_charts = []  # two boundary-oriented, non-root tensors per sentence
full_inside_charts = [] if KEEP_FULL_CHARTS else None

with torch.inference_mode():
    for sample_index in range(N_SAMPLES):
        start_symbol = random.randint(1, n)
        seq, _ = sample_from_grammar(
            sampling_grammar,
            str(start_symbol),
            DEPTH,
            generation_p_2,
            generation_p_3,
        )
        d = len(seq)

        _, inside_tensor_list = parallel_inside_masked(
            seq,
            tensor_bin,
            tensor_ter,
            DEPTH,
            V,
            mask_from=d,
        )

        sentences.append(seq)
        inside_charts.append(charts_to_boundaries(inside_tensor_list, d))
        if KEEP_FULL_CHARTS:
            full_inside_charts.append(
                tuple(
                    chart[:, : d + 1, : d + 1]
                    .to(device="cpu", dtype=STORAGE_DTYPE)
                    .contiguous()
                    for chart in inside_tensor_list
                )
            )

        if (sample_index + 1) % 1024 == 0:
            print(f"generated {sample_index + 1:,}/{N_SAMPLES:,}")

len(sentences), len(inside_charts)
time_2=time.time()
ct=time_2-time_1
print(f"Time taken to generate {N_SAMPLES} samples and their inside charts: {ct:.2f} seconds")

## Inspect and optionally serialize

In [ ]:
example = 0
level_1, level_2 = inside_charts[example]
print("sentence length:", len(sentences[example]))
print("level 1 [token boundary, latent]:", tuple(level_1.shape))
print("level 2 [token boundary, latent]:", tuple(level_2.shape))

In [ ]:
# Uncomment to save. inside_charts contains the compact boundary representation.
# torch.save(
#     {"sentences": sentences, "inside_charts": inside_charts},
#     "validation_set_with_boundary_charts.pt",
# )

## Example: normalized latent probabilities at two boundaries

For one sentence of at least nine tokens, compare the normalized inside mass over the `V` latents at the boundary after token 9 and at the final boundary. Rows show the two latent levels.

In [ ]:
import matplotlib.pyplot as plt

# plot_example = next(
#     index for index, sentence in enumerate(sentences) if len(sentence) >= 9
# )
plot_example=8
plot_sentence = sentences[plot_example]
plot_levels = inside_charts[plot_example]

# Rows of a boundary chart align with tokens: row 8 is the boundary after
# the ninth token, while row -1 is the sentence's final boundary.
selected_boundaries = {"after token 9": 8, "after token 18": 17}
latent_ids = np.arange(1, V + 1)

def normalize_inside(vector):
    total = vector.sum()
    return vector / total if total > 0 else torch.zeros_like(vector)

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey=True)
for level_index, level_chart in enumerate(plot_levels):
    for column, (boundary_name, boundary_index) in enumerate(selected_boundaries.items()):
        probabilities = normalize_inside(level_chart[boundary_index]).numpy()       
        #probabilities =level_chart[boundary_index].numpy()
        ax = axes[level_index, column]
        ax.bar(latent_ids, probabilities)
        ax.set_title(f"Level {level_index + 1}, {boundary_name}")
        ax.set_xticks(latent_ids)
        ax.set_xlabel("latent")
        ax.set_ylabel("normalized inside probability")
        ax.set_ylim(0, 1)

fig.suptitle(
    f"Example {plot_example}: sentence length {len(plot_sentence)}", y=1.02
)
fig.tight_layout()
plt.show()
print("sentence:", plot_sentence)